# Interactive Script: **coregister_cube** and **coregister_cube_roi**

**Author:** Baturalp Arisoy<br>
**Contact:** baturalp.arisoy@uni-wuerzburg.de - Call me Batu :)

## Overview
This notebook guides the user through the essentials of co-registration of Sentinel-2 cube. The user can activate: 
1. **coregister_cube** to automatically co-register all available, analysis worth scenes (recommended function). The code ensures maximum amount of possible co-registration by scanning through the entire scene. 
2. **coregister_cube_roi** by manually selecting a polygon using the interactive leafmap by selecting a known stable area on the surface. This function is much faster than first function, however satellite time-series are complicated and can result removing many analysis worth scenes (scenes with very low cloud percentage). Therefore, the first method is always recommended to sustain the maximum amount of available scenes!

## 1. Auto Co-Registration (Recommended!)

In [1]:
from stac2cube import coregister_cube

In [ ]:
out_ds = coregister_cube(
    input_path="/dss/dsstbyfs02/pr94no/pr94no-dss-0001/drylands/results/aktal_west_masked70.nc",          # can be DataArray, Dataset and NetCDF
    grid_size=7, # If the current setup still removes scenes with low cloud percentages, 
                            #try increasing grid_size. It will take longer to process but could result better.
    max_cc=5,
    time_period= ["2024-03-01", "2024-08-08"],
    min_reliability_keep=10.0,
    min_reliability_update_ref=70.0,
    max_cloud_update_ref=20.0,
    output_path = None           # If None, coregistered file will be exported to same folder of input, with extra prefix "_cr"
)

## 2. Custom ROI Co-Registration

In [ ]:
# Select your polygon on the interactive map and continue with the next cell

import leafmap
import numpy as np
import xarray as xr

stac = xr.open_dataset("/dss/dsstbyfs02/pr94no/pr94no-dss-0001/drylands/results/aktal_west_masked70.nc")
stac = stac.Spectral_Temporal_Stack

xmin, ymin, xmax, ymax = map(float, np.asarray(stac.bbox))

m = leafmap.Map(height="800px")
m.add_basemap("Google Hybrid")

# leafmap expects bounds as [[south, west], [north, east]] i.e. [[ymin, xmin], [ymax, xmax]]
m.fit_bounds([[ymin, xmin], [ymax, xmax]])

m

In [4]:
from stac2cube import coregister_cube_roi

In [ ]:
# roi (leafmap)
polygon_map = m.user_roi["geometry"]
out_ds = coregister_cube_roi(
    input_path= stac,
    roi= polygon_map,
    max_cc= 5,
    time_period= None, #["2024-04-19", "2024-10-30"]
    min_reliability_keep= 10.0,
    min_reliability_update_ref= 40.0,
    max_cloud_update_ref= 20.0,
    output_path= "./results/roi_coregistration.nc",
)